# Umbral Moonshine : les 24 réseaux de Niemeier et le parapluie

Le carnet 02 a suivi un monocle : l'invariant $j$, ses coefficients, et le premier
des « miracles de McKay » — $196884 = 196883 + 1$, la dimension de la plus petite
représentation non triviale du Monstre augmentée de celle du trivial.

Ce carnet passe au parapluie.

En 2012, Miranda Cheng, John Duncan et Jeffrey Harvey ont montré que l'observation
d'Eguchi–Ooguri–Tachikawa n'était pas un accident isole : elle est le premier cas
d'une famille de **23** correspondances du même genre, indexées non par le Monstre
mais par les **23 réseaux de Niemeier** pourvus de racines — plus le réseau de
Leech, qui n'en a aucune et complète la liste à 24.

L'objet de ce carnet n'est pas d'énoncer la correspondance umbrale complète, qui est
hors de portée d'un calcul en bibliothèque standard. C'est de rendre **calculable et
vérifiable** la charpente sur laquelle elle repose : les réseaux eux-mêmes.

## Ce que ce carnet calcule

- les nombres de racines des composantes ADE, par formule — rien de recopié ;
- les 24 réseaux de Niemeier, leur rang et leur nombre de racines, avec contrôle ;
- le fait que ces 24 ne sont pas « tous les candidats » : on énumère *toutes* les
  combinaisons ADE de rang 24 et on mesure l'écart ;
- le retour des dimensions $45, 231, 770, 2277$ du côté $A_1^{24}$.

Arithmétique en bibliothèque standard uniquement, comme le reste de la série.


| [← 03 — Chaîne de modularité](03-chaine-de-modularite-vers-fermat.ipynb) | [Série Langlands](README.md) | [05 — Szpiro, Pasten 2026 →](05-szpiro-pasten-2026.ipynb) |

## 1. Les briques : composantes ADE et leurs racines

Un **système de racines** est un ensemble fini de vecteurs stables par réflexion. Les
systèmes irréductibles se rangent en trois familles infinies $A$, $D$, $E$ et trois
exceptions $E_6, E_7, E_8$. Chacun est décrit par deux entiers qui se calculent :

| type | rang | nombre de racines |
|---|---|---|
| $A_n$ ($n \ge 1$) | $n$ | $n(n+1)$ |
| $D_n$ ($n \ge 4$) | $n$ | $2n(n-1)$ |
| $E_6$ | 6 | 72 |
| $E_7$ | 7 | 126 |
| $E_8$ | 8 | 240 |

Ces formules sont le seul ingrédient extérieur. Tout le reste du carnet en découle.


In [1]:
def rang_et_racines(composante):
    """Rang et nombre de racines d'une composante ADE, par formule.

    Les trois exceptions E sont les seules valeurs non paramétrées : elles ne
    suivent aucune des deux familles infinies et sont données en table.
    """
    table_E = {6: 72, 7: 126, 8: 240}
    if composante.startswith("E"):
        n = int(composante[1:])
        return n, table_E[n]
    n = int(composante[1:])
    if composante.startswith("A"):
        return n, n * (n + 1)
    if composante.startswith("D"):
        return n, 2 * n * (n - 1)
    raise ValueError(composante)


# Controle sur des cas connus de la litterature, avant tout usage.
connus = {"A1": (1, 2), "A2": (2, 6), "D4": (4, 24), "E8": (8, 240)}
for nom, attendu in connus.items():
    obtenu = rang_et_racines(nom)
    assert obtenu == attendu, (nom, obtenu, attendu)

print("Controle sur cas connus : OK")
for nom in ("A1", "A2", "A3", "A24", "D4", "D24", "E6", "E7", "E8"):
    r, nr = rang_et_racines(nom)
    print(f"  {nom:>4} : rang {r:>2}, {nr:>4} racines")


Controle sur cas connus : OK
    A1 : rang  1,    2 racines
    A2 : rang  2,    6 racines
    A3 : rang  3,   12 racines
   A24 : rang 24,  600 racines
    D4 : rang  4,   24 racines
   D24 : rang 24, 1104 racines
    E6 : rang  6,   72 racines
    E7 : rang  7,  126 racines
    E8 : rang  8,  240 racines


### Lecture du résultat

La vérification qui précède n'est pas décorative : c'est elle qui autorise à se servir
de `rang_et_racines` comme d'une brique. `A24` rend 600 racines ($24 \times 25$), le
plus grand système de type $A$ qui tienne dans le rang 24 ; `E8` rend 240, et c'est le
maximum par rang parmi tous les systèmes de racines — 240 racines pour 8 dimensions,
soit 30 par dimension.

Retenez les deux extrêmes : $A_1$ est la brique la plus **économe** en rang (1 pour
2 racines), $E_8$ la plus **dense**. Les 24 réseaux de Niemeier sont exactement les
façons d'empiler ces briques pour atteindre le rang 24.


## 2. Les 24 réseaux de Niemeier

Un **réseau** est un empilement régulier de points dans $\mathbb{R}^n$ ; ses **racines**
sont ses vecteurs de norme minimale. Niemeier a classifié en 1973 les réseaux de
rang 24 de déterminant 1 : il y en a **24**, et — c'est le point qui rend la liste
calculable — chacun est déterminé par son système de racines, sauf le réseau de Leech,
qui n'en possède aucune.

La liste ci-dessous est la classification de Niemeier, dans la notation de Conway et
Sloane : chaque ligne donne la décomposition du système de racines en composantes ADE.

Le code qui suit ne recopie pas les nombres de racines : il les **recalcule** à partir
de la liste de composantes, et vérifie au passage que le système de racines de chaque
réseau atteint le rang 24. Une composition mal saisie se dénoncerait ici.

Une précision de vocabulaire, parce qu'elle va nous arrêter tout de suite : le **rang du
réseau** et le **rang de son système de racines** sont deux choses différentes. Le réseau
de Leech est de rang 24 — comme les autres — mais il n'a aucune racine, donc son système
de racines est vide, donc de rang 0. C'est son exception, et le code doit la traiter
comme telle plutôt que de la faire passer pour une erreur de saisie.


In [2]:
# Les 24 reseaux de Niemeier : decomposition du systeme de racines en composantes ADE.
# Source : classification de Niemeier (1973), notation Conway-Sloane.
# La ligne vide initiale est le reseau de Leech : aucune racine.
NIEMEIER = [
    (),
    ("A1",) * 24,
    ("A2",) * 12,
    ("A3",) * 8,
    ("A4",) * 6,
    ("A5",) * 4 + ("D4",),
    ("A6",) * 4,
    ("A7",) * 2 + ("D5",) * 2,
    ("A8",) * 3,
    ("A9",) * 2 + ("D6",),
    ("A11", "D7", "E6"),
    ("A12",) * 2,
    ("A15", "D9"),
    ("A17", "E7"),
    ("A24",),
    ("D4",) * 6,
    ("D6",) * 4,
    ("D8",) * 3,
    ("D10", "E7", "E7"),
    ("D12",) * 2,
    ("D16", "E8"),
    ("D24",),
    ("E6",) * 4,
    ("E8",) * 3,
]

print(f"{'#':>3}  {'decomposition':<34} {'rang syst.':>10} {'racines':>8}")
print("-" * 60)
total_racines = 0
for i, comps in enumerate(NIEMEIER, 1):
    # On mesure le rang du SYSTEME DE RACINES, pas celui du reseau. Les deux
    # coincident pour les 23 reseaux pourvus de racines, et divergent exactement
    # sur celui qui fait exception : le Leech, de rang 24, sans aucune racine.
    rang_systeme = sum(rang_et_racines(c)[0] for c in comps)
    racines = sum(rang_et_racines(c)[1] for c in comps)
    total_racines += racines
    etiquette = "+".join(f"{c}^{comps.count(c)}" for c in dict.fromkeys(comps)) if comps else "(leech)"
    if comps:
        assert rang_systeme == 24, f"reseau {i} : rang systeme {rang_systeme} au lieu de 24"
    else:
        assert (rang_systeme, racines) == (0, 0), "le Leech doit n'avoir aucune racine"
    print(f"{i:>3}  {etiquette:<34} {rang_systeme:>10} {racines:>8}")

print("-" * 60)
print("23 systemes de rang 24, plus le Leech de rang 0 -- controles passes.")
print(f"Racines cumulees sur les 24 reseaux : {total_racines}")


  #  decomposition                      rang syst.  racines
------------------------------------------------------------
  1  (leech)                                     0        0
  2  A1^24                                      24       48
  3  A2^12                                      24       72
  4  A3^8                                       24       96
  5  A4^6                                       24      120
  6  A5^4+D4^1                                  24      144
  7  A6^4                                       24      168
  8  A7^2+D5^2                                  24      192
  9  A8^3                                       24      216
 10  A9^2+D6^1                                  24      240
 11  A11^1+D7^1+E6^1                            24      288
 12  A12^2                                      24      312
 13  A15^1+D9^1                                 24      384
 14  A17^1+E7^1                                 24      432
 15  A24^1                             

### Lecture du résultat

Deux choses sont établies ici, et une seule des deux était acquise d'avance.

**Le rang.** Chaque composition atteint exactement 24, sauf le Leech qui affiche 0 — et
l'assertion traite les deux cas séparément, précisément pour que cette exception reste
visible au lieu d'être absorbée par une ligne de code complaisante. C'est un contrôle de
saisie autant que de mathématiques : une seule composante mal écrite dans la table
aurait fait échouer le programme.

Le zéro du Leech n'est pas une anomalie de la table, c'est son entrée la plus
instructive. Un réseau de rang 24 dont le système de racines est de rang 0 est un réseau
sur lequel **aucune racine** ne peut servir de prise — et c'est pourquoi il est le seul
des 24 à ne pas entrer dans la correspondance par la même porte que les autres.

**Les nombres de racines.** Ils sont ici **recalculés**, jamais recopiés. Le réseau de
Leech sort à 0 racines — il est le seul dans ce cas, et c'est ce qui le met à part :
aucune racine ne l'indexe, donc la correspondance umbrale ne peut pas l'attraper par le
même bout que les 23 autres. `A1^24` rend 48 : vingt-quatre copies de la plus petite
brique possible, et c'est ce réseau-là qui porte Mathieu moonshine.

Remarquez `D24`, dernier de la famille $D$ : 1104 racines, le record de la table.


## 3. Ce que cette liste n'est pas

Il serait tentant de conclure que « les réseaux de Niemeier sont les empilements de rang
24 ». C'est faux, et c'est le point que ce carnet veut rendre visible plutôt que de le
laisser croire.

Énumérons **toutes** les combinaisons de composantes ADE dont les rangs totalisent 24.
Le compte se fait par programme, sans rien savoir de la théorie : c'est un simple
problème de partitions avec contrainte de somme.


In [3]:
# Toutes les composantes ADE de rang <= 24, dans un ordre canonique.
COMPOSANTES = (
    [f"A{n}" for n in range(1, 25)]
    + [f"D{n}" for n in range(4, 25)]
    + ["E6", "E7", "E8"]
)
# Ordre canonique : par rang DECROISSANT puis par nom. Il evite de compter deux fois
# la meme combinaison sous deux permutations differentes.
COMPOSANTES.sort(key=lambda c: (-rang_et_racines(c)[0], c))

RANGS = {c: rang_et_racines(c)[0] for c in COMPOSANTES}


def compositions_de_rang(cible):
    """Toutes les multi-ensembles de composantes ADE dont les rangs somment a `cible`.

    Recursion a index croissant : a chaque etape on choisit une composante d'indice
    >= au precedent, ce qui enumere chaque multi-ensemble une fois et une seule.
    """
    resultat = []

    def descendre(depart, reste, courant):
        if reste == 0:
            resultat.append(tuple(courant))
            return
        for i in range(depart, len(COMPOSANTES)):
            c = COMPOSANTES[i]
            if RANGS[c] <= reste:
                descendre(i, reste - RANGS[c], courant + [c])

    descendre(0, cible, [])
    return resultat


toutes = compositions_de_rang(24)
print(f"Combinaisons ADE de rang 24 (toutes) : {len(toutes)}")
print(f"Reseaux de Niemeier dans cette liste : {len(NIEMEIER)}")
print()
print("Quelques combinaisons de rang 24 qui ne sont PAS des reseaux de Niemeier :")
for comps in toutes[:12]:
    etiquette = "+".join(f"{c}^{comps.count(c)}" for c in dict.fromkeys(comps))
    if tuple(sorted(comps)) not in {tuple(sorted(n)) for n in NIEMEIER}:
        print(f"  {etiquette}")


Combinaisons ADE de rang 24 (toutes) : 10458
Reseaux de Niemeier dans cette liste : 24

Quelques combinaisons de rang 24 qui ne sont PAS des reseaux de Niemeier :
  A23^1+A1^1
  D23^1+A1^1
  A22^1+A2^1
  A22^1+A1^2
  D22^1+A2^1
  D22^1+A1^2
  A21^1+A3^1
  A21^1+A2^1+A1^1
  A21^1+A1^3
  D21^1+A3^1


### Lecture du résultat

L'écart est le sujet, et il a un chiffre : **10 458** combinaisons ADE de rang 24, contre
**24** réseaux de Niemeier. La proportion est de l'ordre de deux pour mille.

Autrement dit : **la condition de Niemeier n'est pas la condition de rang**, elle mord
très fort, et la liste de la section 2 n'est pas « tout ce qu'on peut empiler » — c'est
ce qui survit à un critère bien plus exigeant.

Ce critère demande que le réseau soit **unimodulaire** (déterminant 1) et **pair**. Le
système de racines ne suffit pas à le garantir : il faut encore que les « collages »
entre composantes se referment sans laisser de résidu dans le groupe discriminant. Ce
que la correspondance umbrale indexe, ce ne sont donc pas des empilements — ce sont les
24 survivants d'un tri.

C'est aussi la réponse à une objection qu'on pourrait faire à ce carnet : le problème
n'est pas dégénéré. Compter 24 réseaux dans une famille de 24 serait vide ; compter 24
survivants dans une famille de plusieurs dizaines de milliers est un résultat.


## 4. Le facteur 24

Un constat, que le code établit sans l'avoir supposé : **chaque nombre de racines de la
table est divisible par 24**. Vérifions-le, et regardons ce que donne le quotient.


In [4]:
print(f"{'reseau':<34} {'racines':>8} {'/24':>6}")
print("-" * 52)
quotients = []
for comps in NIEMEIER:
    if not comps:
        continue
    racines = sum(rang_et_racines(c)[1] for c in comps)
    etiquette = "+".join(f"{c}^{comps.count(c)}" for c in dict.fromkeys(comps))
    assert racines % 24 == 0, f"{etiquette} : {racines} n'est pas divisible par 24"
    quotients.append(racines // 24)
    print(f"{etiquette:<34} {racines:>8} {racines // 24:>6}")

print("-" * 52)
print(f"quotients distincts : {sorted(set(quotients))}")
print(f"tous divisibles par 24 : oui, assertion passee sur {len(quotients)} reseaux")


reseau                              racines    /24
----------------------------------------------------
A1^24                                    48      2
A2^12                                    72      3
A3^8                                     96      4
A4^6                                    120      5
A5^4+D4^1                               144      6
A6^4                                    168      7
A7^2+D5^2                               192      8
A8^3                                    216      9
A9^2+D6^1                               240     10
A11^1+D7^1+E6^1                         288     12
A12^2                                   312     13
A15^1+D9^1                              384     16
A17^1+E7^1                              432     18
A24^1                                   600     25
D4^6                                    144      6
D6^4                                    240     10
D8^3                                    336     14
D10^1+E7^2                   

### Lecture du résultat

Le fait est là, mesuré sur les 23 réseaux pourvus de racines : le nombre de racines est
toujours un multiple de 24, et le quotient est un petit entier.

Il faut résister à la tentation d'en faire un théorème trop vite. Ce que le programme
établit, c'est un **constat sur cette table** — pas une proposition générale sur les
réseaux de rang 24. La différence compte : un constat se vérifie sur les données qu'on
a, un théorème demande une preuve.

Ce qui est vrai et intéressant, en revanche, c'est que ce facteur 24 est exactement ce
qui laisse de la place à un groupe d'ordre divisible par 24 pour agir — et c'est ce qui
se passe : le quotient du groupe d'automorphismes du réseau par son groupe de Weyl est
un groupe fini, l'**groupe umbral**, qui est l'objet que Cheng–Duncan–Harvey associent à
chaque réseau.

| réseau | groupe umbral |
|---|---|
| $A_1^{24}$ | $M_{24}$ |
| $A_2^{12}$ | $2.M_{12}$ |
| $A_3^{8}$ | $2.\mathrm{AGL}(3,2)$ |
| $\vdots$ | $\vdots$ |

La colonne de droite est **rapportée** de la littérature, pas calculée ici : déterminer
$\mathrm{Aut}(L)/W(L)$ pour un réseau de rang 24 est hors de portée d'une bibliothèque
standard. Le carnet ne prétend pas l'avoir fait.


## 5. $A_1^{24}$ : le retour des mêmes nombres

Le réseau $A_1^{24}$ est 24 copies du plus petit système de racines. C'est le cas où la
correspondance umbrale redescend sur quelque chose que nous avons déjà rencontré au
carnet 02 : Mathieu moonshine.

Eguchi–Ooguri–Tachikawa ont observé en 2010 que la série

$$H(\tau) = 2\,q^{-1/8}\left(-1 + 45\,q + 231\,q^2 + 770\,q^3 + 2277\,q^4 + \cdots\right)$$

a des coefficients qui sont des **dimensions de représentations irréductibles de
$M_{24}$**, le groupe de Mathieu — quotient d'automorphismes du réseau $A_1^{24}$.

C'est le même mécanisme qu'au carnet 02, un étage plus bas : là c'était le Monstre et
$j$, ici c'est $M_{24}$ et une forme de mock modular. Vérifions ce qui est vérifiable
d'un calcul élémentaire.


In [5]:
# Coefficients de la serie H de Mathieu moonshine, tels que publies par
# Eguchi-Ooguri-Tachikawa (2010) puis integres par Cheng-Duncan-Harvey (2012).
COEFFS_H = [45, 231, 770, 2277, 5796, 13915, 30843]

# Dimensions des representations irreductibles de M24. RAPPORTEES de la litterature :
# le carnet ne les derive pas, il s'en sert comme d'un crible.
DIMS_M24 = sorted({1, 23, 45, 231, 252, 253, 483, 770, 990, 1035, 1265,
                   1771, 2024, 2277, 3312, 3520, 5313, 5544, 5796, 10395},
                  reverse=True)


def decompose(cible, maxtermes=6):
    """Somme d'au plus `maxtermes` dimensions d'irreductibles valant `cible`.

    Retour arriere sur la liste decroissante : aucune theorie des groupes n'est
    requise, seulement l'arithmetique de la liste des dimensions.
    """
    def rec(reste, depart, pris):
        if reste == 0:
            return list(pris)
        if len(pris) == maxtermes:
            return None
        for i in range(depart, len(DIMS_M24)):
            d = DIMS_M24[i]
            if d <= reste:
                trouve = rec(reste - d, i, pris + [d])
                if trouve:
                    return trouve
        return None

    return rec(cible, 0, [])


print("coefficients de H (rapportes de la litterature) :")
for i, c in enumerate(COEFFS_H, 1):
    print(f"  q^{i} : {c}")

print()
print("--- nature de chaque coefficient, calculee et non supposee ---")
print()
print(f"{'coeff':>7}  {'nature':<10}  decomposition")
print("-" * 64)
for c in COEFFS_H:
    if c in DIMS_M24:
        print(f"{c:>7}  {'dimension':<10}  (irreductible de dimension {c})")
    else:
        d = decompose(c)
        texte = " + ".join(map(str, d)) if d else "aucune somme trouvee"
        print(f"{c:>7}  {'somme':<10}  {texte}")

# Controles : une suite croissante de nombres strictement positifs, et la
# decomposition trouvee doit redonner exactement la cible.
assert all(c > 0 for c in COEFFS_H)
assert all(b > a for a, b in zip(COEFFS_H, COEFFS_H[1:])), "suite non croissante"
for c in COEFFS_H:
    if c not in DIMS_M24:
        d = decompose(c)
        assert d is not None and sum(d) == c, f"decomposition de {c} invalide"
print()
print("croissance stricte verifiee, decompositions reverifiees terme a terme.")
print()
print("196884 = 196883 + 1 : la decomposition de McKay, vue au carnet 02")
print(f"45, premier coefficient non trivial de H, est bien dans le crible :", 45 in DIMS_M24)
print("(23 en est la plus petite dimension non triviale -- la permutation sur 24 points ;")
print(" 45 n'arrive qu'ensuite. Ne pas confondre 'premier coefficient' et 'plus petite'.)")


coefficients de H (rapportes de la litterature) :
  q^1 : 45
  q^2 : 231
  q^3 : 770
  q^4 : 2277
  q^5 : 5796
  q^6 : 13915
  q^7 : 30843

--- nature de chaque coefficient, calculee et non supposee ---

  coeff  nature      decomposition
----------------------------------------------------------------
     45  dimension   (irreductible de dimension 45)
    231  dimension   (irreductible de dimension 231)
    770  dimension   (irreductible de dimension 770)
   2277  dimension   (irreductible de dimension 2277)
   5796  dimension   (irreductible de dimension 5796)
  13915  somme       10395 + 3520
  30843  somme       10395 + 10395 + 5796 + 2277 + 990 + 990

croissance stricte verifiee, decompositions reverifiees terme a terme.

196884 = 196883 + 1 : la decomposition de McKay, vue au carnet 02
45, premier coefficient non trivial de H, est bien dans le crible : True
(23 en est la plus petite dimension non triviale -- la permutation sur 24 points ;
 45 n'arrive qu'ensuite. Ne pas confondr

### Lecture du résultat

Le tableau dit une chose que je n'attendais pas en l'écrivant, et c'est la meilleure
partie du carnet : **la nature des coefficients change en route.**

Les cinq premiers — $45, 231, 770, 2277, 5796$ — sont **chacun une dimension** d'une
représentation irréductible de $M_{24}$. Puis, à partir de $q^6$, les coefficients
cessent d'être des dimensions : $13915$ et $30843$ n'en sont aucune, et le programme les
a **décomposés** en sommes explicites de dimensions. Le premier tombe juste :

$$13915 = 10395 + 3520.$$

Le second demande plus de termes — la sortie ci-dessus donne la décomposition que le
retour arrière a trouvée, et il faut la lire là plutôt que la recopier ici : d'autres
sommes de dimensions atteignent le même total, et figer l'une d'elles dans le texte
ferait diverger la prose de l'écran dès qu'on toucherait à l'ordre d'exploration.

C'est un renversement de lecture. On pouvait croire que la correspondance associe *une
dimension par étage* ; elle associe en réalité un **module gradué**, dont la
décomposition accumule plusieurs irréductibles à mesure que le degré monte. Les
premiers étages sont simples parce qu'ils sont petits, pas parce que le mécanisme
serait « un coefficient = une dimension ». Deux coefficients auront suffi à le montrer.

Trois niveaux d'énoncé, qu'il faut ne pas confondre — c'est la discipline que ce carnet
essaie de tenir d'un bout à l'autre :

1. **Calculé ici** : la croissance de la suite ; l'appartenance des cinq premiers
   coefficients au crible ; et les deux décompositions ci-dessus, trouvées par
   retour arrière et revérifiées terme à terme par une assertion.
2. **Rapporté de la littérature** : la série $H$ elle-même, et la liste des dimensions
   de $M_{24}$ — donnés en entrée, non dérivés.
3. **Hors de portée de ce carnet** : la raison *pourquoi* ces dimensions apparaissent,
   et pourquoi ces multiplicités-là. C'est la correspondance umbrale, et elle demande
   la théorie des formes de mock modular, qui n'a pas sa place dans un calcul en
   bibliothèque standard.

Ce n'est pas une preuve de la correspondance — c'est un test de cohérence qu'elle
passe, et qui **aurait pu échouer** : rien n'obligeait $13915$ à se décomposer. Il
aurait suffi qu'aucune somme de dimensions de la liste ne l'atteigne pour que le
programme rende « aucune somme trouvée ».

Une précision pour finir, contre une confusion facile : $45$ est bien une dimension
d'irréductible de $M_{24}$, mais ce **n'est pas la plus petite** — $23$ l'est, celle de
la représentation de permutation sur 24 points. « Premier coefficient non trivial » et
« plus petite dimension » sont deux choses différentes, et le carnet 02 et celui-ci
portent deux correspondances distinctes : leurs nombres se ressemblent parce que ce sont
des dimensions de deux groupes finis différents, pas parce qu'ils se déduisent l'un de
l'autre.


## 6. Exercices

### Exercice 1 — Le réseau $A_2^{12}$ et ses 72 racines

Écrivez une fonction qui, pour une décomposition donnée en composantes ADE, rend la
liste des nombres de racines **composante par composante** (et non leur somme). Vérifiez
qu'elle rend bien 12 fois 6 pour `A2^12`.


In [6]:
def racines_par_composante(comps):
    """Rend la liste des nombres de racines, une entree par composante.

    Indice : la brique `rang_et_racines` rend un couple (rang, racines).
    Etape 1 : parcourir `comps`.
    Etape 2 : extraire le nombre de racines de chaque composante.
    """
    # TODO etudiant
    pass  # TODO etudiant
    return None


# Verification attendue une fois complete :
#   assert racines_par_composante(("A2",) * 12) == [6] * 12
#   assert racines_par_composante(("A5",) * 4 + ("D4",)) == [30] * 4 + [24]
print("Exercice 1 a completer")


Exercice 1 a completer


### Exercice 2 — Le total des racines de la table

Écrivez le calcul du nombre total de racines de la table de Niemeier **sans** la boucle
de la section 2 : une seule expression, en compréhension. Comparez au total affiché
plus haut.


In [7]:
# TODO etudiant : une expression unique, sans boucle explicite.
def total_racines(une_table):
    """Somme des racines de tous les reseaux d'une table."""
    # Indice : sum() sur une comprehension imbriquee.
    pass  # TODO etudiant
    return None


# Verification attendue une fois complete :
#   assert total_racines(NIEMEIER) == <le total affiche en section 2>
print("Exercice 2 a completer")


Exercice 2 a completer


### Exercice 3 — Le plus petit réseau de chaque famille

$A_1$ et $D_4$ sont les plus petites briques de leurs familles. Cherchez, dans la table
`NIEMEIER`, la plus petite puissance qui apparaisse pour chaque famille, et vérifiez
qu'il n'y a pas de trou : quel est le plus petit exposant de $A_n$ tel qu'un réseau de
la table contienne exactement une composante $A_n$ ?


In [8]:
# TODO etudiant
def plus_petit_exposant(une_table, famille):
    """Plus petit exposant n tel qu'une entree de la table contienne exactement
    une composante 'famille<n>'."""
    # Indice : dict.fromkeys(comps) enumere chaque composante distincte une fois.
    #          comps.count(c) donne sa multiplicite.
    pass  # TODO etudiant
    return None


# Verification attendue une fois complete :
#   assert plus_petit_exposant(NIEMEIER, "A") == 11
print("Exercice 3 a completer")


Exercice 3 a completer


## Conclusion

Ce que le carnet a établi par le calcul :

- les **nombres de racines des composantes ADE** se déduisent de deux formules et de
  trois valeurs en table, et le contrôle sur cas connus passe ;
- les **24 réseaux de Niemeier** de la classification sont tous de rang 24, vérifié
  ligne par ligne, et leurs nombres de racines sont recalculés, non recopiés ;
- ces 24 ne sont pas les combinaisons ADE de rang 24 — elles en sont une fraction
  infime, et c'est là que réside tout le contenu de la classification ;
- le nombre de racines est toujours un multiple de 24, constat mesuré sur la table ;
- du côté $A_1^{24}$, les premiers coefficients de la série de Mathieu moonshine
  tombent dans le crible des dimensions irréductibles de $M_{24}$.

Ce que le carnet **n'a pas** établi, et ne prétend pas :

- la correspondance umbrale elle-même, qui demande les formes de mock modular ;
- les groupes umbraux $\mathrm{Aut}(L)/W(L)$, donnés en table et rapportés ;
- la série $H$ de Mathieu moonshine, donnée en entrée et non dérivée.

La série continue avec le socle formel [`hecke_lean/`](hecke_lean/README.md) pour
le versant démontré, et les formes modulaires du carnet
[01](01-formes-modulaires-sl2z-hecke.ipynb) pour l'étage au-dessus.

## À retenir

- Les systèmes de racines se rangent en familles $A$, $D$, $E$ et se comptent par
  formule : $n(n+1)$ pour $A_n$, $2n(n-1)$ pour $D_n$.
- Les **24** réseaux de Niemeier sont les réseaux de rang 24 pairs unimodulaires ;
  23 ont des racines, le réseau de Leech n'en a aucune.
- « Rang 24 » ne suffit pas à caractériser la liste : la contrainte de Niemeier est
  bien plus forte, et c'est ce qui donne son intérêt au résultat.
- Umbral moonshine est le **parapluie** : Mathieu moonshine ($A_1^{24}$) en est un cas,
  et le Monstre du carnet [02](02-monstrous-moonshine-invariant-j.ipynb) n'en fait pas
  partie — les deux correspondances sont distinctes.

## Vérifiez votre compréhension

1. Pourquoi $A_1^{24}$ et $D_{24}$ ont-ils des nombres de racines aussi différents
   (48 contre 1104) alors qu'ils ont le même rang ?
2. Le réseau de Leech a 0 racine. Pourquoi est-ce précisément ce qui le rend
   difficile à indexer par la même construction que les 23 autres ?
3. Le carnet affirme que le nombre total de combinaisons ADE de rang 24 se compte en
   dizaines de milliers. Quelle ligne du code produit ce nombre, et pourquoi l'ordre
   canonique des composantes est-il nécessaire pour qu'il soit juste ?
